# ⚠️ Run this cell first before running the other cells
## Helper Methods

In [1]:
import spacy
import en_core_web_sm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nlp = en_core_web_sm.load()
crypto_custom_weights = {'adoption': 3.5, 'integration': 3.0, 'merchant': 2.5, 'exemption': 2.5, 'enshrining': 2.5, 'rollout': 2.5, 'expansion': 2.5, 'sovereign': 3.5, 'treasury': 3.0, 'reserves': 3.0, 'accumulation': 2.5, 'breakout': 3.5, 'rebound': 2.5, 'inflow': 3.0, 'outflow': -3.5, 'liquidated': -3.5, 'capitulation': -4.0, 'oversold': 2.0, 'correction': -1.5, 'upgrade': 3.0, 'mainnet': 2.5, 'scalability': 2.0, 'exploit': -4.0, 'outage': -3.0, 'halt': -3.0, 'boom': 4.0, 'eyes': 1.0, 'rally': 3.0, 'retreat': -2.0, 'slide': -2.0, 'sink': -3.5, 'plunge': -4.0, 'tumble': -3.5, 'bloodbath': -4.0, 'moon': 4.0, 'pump': 3.5, 'bullish': 3.5, 'ath': 3.0, 'acquire': 3.0, 'primed': 2.0, 'overperform': 3.0, 'outperform': 3.0, 'rug': -4.0, 'dump': -4.0, 'bearish': -3.5, 'sec': -2.0, 'underperform': -3.0, 'lag': -2.5, 'fud': -2.5, 'hack': -4.0, 'incline': 2.0, 'decline': -2.5, 'surge': 3.5, 'stake': 2.5, 'optimism': 2.0, 'fade': -3.0, 'erase': -3.0, 'shrink': -2.0, 'diverge': -1.5, 'slip': -2.0, 'liquidation': -3.5}
analyzer = SentimentIntensityAnalyzer()
analyzer.lexicon.update(crypto_custom_weights)

def lemmatize_text(text):
    doc = nlp(text)
    lemm_text = " ".join([token.lemma_ for token in doc])
    return lemm_text

def sentiment_analysis(lemm_text):
    crypto_custom_weights = {'adoption': 3.5, 'integration': 3.0, 'merchant': 2.5, 'exemption': 2.5, 'enshrine': 2.5, 'rollout': 2.5, 'expansion': 2.5, 'sovereign': 3.5, 'treasury': 3.0, 'reserve': 3.0, 'accumulation': 2.5, 'breakout': 3.5, 'rebound': 2.5, 'inflow': 3.0, 'outflow': -3.5, 'liquidate': -3.5, 'capitulation': -4.0, 'oversold': 2.0, 'correction': -1.5, 'upgrade': 3.0, 'mainnet': 2.5, 'scalability': 2.0, 'exploit': -4.0, 'outage': -3.0, 'halt': -3.0, 'boom': 4.0, 'eye': 1.0, 'rally': 3.0, 'retreat': -2.0, 'slide': -2.0, 'sink': -3.5, 'plunge': -4.0, 'tumble': -3.5, 'bloodbath': -4.0, 'moon': 4.0, 'pump': 3.5, 'bullish': 3.5, 'ath': 3.0, 'acquire': 3.0, 'primed': 2.0, 'overperform': 3.0, 'outperform': 3.0, 'rug': -4.0, 'dump': -4.0, 'bearish': -3.5, 'sec': -2.0, 'underperform': -3.0, 'lag': -2.5, 'fud': -2.5, 'hack': -4.0, 'incline': 2.0, 'decline': -2.5, 'surge': 3.5, 'stake': 2.5, 'optimism': 2.0, 'fade': -3.0, 'erase': -3.0, 'shrink': -2.0, 'diverge': -1.5, 'slip': -2.0, 'liquidation': -3.5}

    # vader model keeps the weight from -4 to 4
    analyzer = SentimentIntensityAnalyzer()
    analyzer.lexicon.update(crypto_custom_weights)
    vs = analyzer.polarity_scores(lemm_text.lower())
    sentiment_score = vs['compound'] # this will return the compounded score from -1 to 1
    return sentiment_score

def is_question(text):
    text_lower = text.lower()
    if text_lower.endswith('?') : return True

    question_starters = ['is','will','should','can']
    if text_lower.split()[0] in question_starters: return True

    panic_triggers =  ['sink', 'plunge', 'tumble', 'liquidated', 'dump', 'bloodbath']
    panic_word_pattern = r'\b('+'|'.join(panic_triggers)+r')\b'
    if re.search(panic_word_pattern,text_lower): return False

    # Checks for standalone words only
    info_keywords = [
            'how', 'why', 'what', 'thoughts', 'opinions', 'guide', 'help',
            'question', 'tutorial', 'technical', 'understand',
            'explain', 'recommendation', 'portfolio', 'comparison'
        ]
    keywords_re = "|".join(info_keywords)
    if re.search(rf'^(?:\s*\S+){0,4}\s*\b({keywords_re})\b', text_lower):
        return True

    return False

def get_symbol(text):
    patten = r"\b(bitcoin|ethereum|ether|dogecoin|btc|eth|doge|solana|sol|xrp|ripple|bnb|ada|cardano|dot|polkadot|litecoin|ltc|chainlink|link|ftx|luna|terra|avax|avalanche|matic|polygon)\b"
    symbols = re.findall(patten,text,re.IGNORECASE)
    return symbols[0].lower() if symbols else "market"

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

# --- stage 1.1 (1/4): extraction of cc prices of 6 months ---

### using apis
##### for binance api, the limit is weight based, 500 candlestick has same weight for 1hr,4hr,...etc., 1hr will get a lot messy(fluctuating) data and more  long intervals will not get the trainable data hence, 4hr is a 'sweet spot' in between


In [4]:
import requests
import mysql.connector
import pandas as pd
from bs4 import BeautifulSoup

# --- Getting the bitcoin data of last 6 Months interval: 4 hours ---
parameters={
    "symbol":"BTCUSDT",
    "interval":"4h",
    "limit": 500
}
response = requests.get(f"https://api.binance.com/api/v3/klines",params=parameters).json()
# get function is the GET request from https
# url takes the get function to the binance k-lines(candlestick) end-point
# params: instead of writing a long url; it straight away requests
# how it does it:
#   1. The params=params argument tells Python to:
#   2. Add a ? at the end of the base url.
#   3. Take every Key and Value from your dictionary.
#   4. Join them with an = sign.
#   5. Separate multiple pairs with an & symbol.
# the 'get' function will return the response from the server(ref. client-server arc response) along with raw data(wiz. json)
# .json will return the .json from the request

print(response)

symboldict = {"BTCUSDT":"btc","ETHUSDT":"eth","SOLUSDT":"sol","DOGEUSDT":"doge","XRPUSDT":"xrp"}

config = {'host':'localhost','user':'root','password':'','database':'crypto_radar_db'}
conn = mysql.connector.connect(**config)
curse = conn.cursor()

try:
    print("🧹cleaning and uploading prices to database")
    dfi = pd.DataFrame(response)
    dfi = dfi[[0,1,2,3,4,7]]
    dfi.columns = ['timestamp_ms','open','high','low','close','volume_usdt']
    dfi['db_time'] = pd.to_datetime(dfi['timestamp_ms'],unit='ms').dt.strftime('%Y-%m-%d %H:%M:%S')


    batch_data = []
    for _, row in dfi.iterrows():
        batch_data.append((symboldict.get(parameters.get("symbol")),row['db_time'], float(row['close']), float(row['volume_usdt'])))

    # INSERT IGNORE avoids duplicates if you run the script twice
    sql = "INSERT IGNORE INTO  coin_prices(asset,timestamp, price_close, volume_usdt) VALUES (%s,%s, %s, %s)"
    curse.executemany(sql, batch_data)
    conn.commit()
except Exception as e:
    print("❌ Unable to upload to database")
    print(e)
finally:
    curse.close()
    conn.close()



[[1766203200000, '88214.98000000', '88573.07000000', '88123.07000000', '88275.41000000', '797.87192000', 1766217599999, '70465230.87775860', 199607, '370.18006000', '32694849.07814550', '0'], [1766217600000, '88275.41000000', '88428.00000000', '88107.35000000', '88277.66000000', '821.72303000', 1766231999999, '72520013.04052370', 186262, '330.42295000', '29161839.93509600', '0'], [1766232000000, '88277.66000000', '88343.73000000', '87795.76000000', '88178.71000000', '1315.67537000', 1766246399999, '115955453.27595540', 271332, '559.81615000', '49341498.95854720', '0'], [1766246400000, '88178.72000000', '88418.73000000', '88103.54000000', '88232.92000000', '574.10491000', 1766260799999, '50646063.75424630', 146754, '275.81192000', '24333152.68544030', '0'], [1766260800000, '88232.92000000', '88443.44000000', '88154.06000000', '88360.90000000', '629.30956000', 1766275199999, '55551786.91834000', 146283, '336.14911000', '29672787.24625560', '0'], [1766275200000, '88360.91000000', '88433.6

# --- stage 1.2 (2/4): extraction of usd prices ---
### using library
- usually bitcoin is traded against dollar
- DXY is symbol of dollar index yahoo uses and nyb means New York Board of trade
- we are tracking DX-Y.NYB. This index measures the Dollar against six major world currencies (Euro, Yen, Pound, etc.).
- DXY < 100: The Dollar is weak. This is usually "Rocket Fuel" for Bitcoin.
- DXY > 105: The Dollar is very strong. This is a "Weight" that pulls the crypto market down.

In [ ]:
import yfinance

# --- Getting the data of USD for last 5 Days on intervals of 1 Hrs ---
ticker = yfinance.Ticker("DX-Y.NYB")
df = ticker.history(period="5d",interval="1h")
print(df)
config = {'host':'localhost','user':'root','password':'','database':'crypto_radar_db'}
conn = mysql.connector.connect(**config)
curse = conn.cursor()

df = ticker.history(period="1d",interval="1h")
print(df)
df.index = df.index.tz_localize(None)
try:
    #fix
    df.index.name = "timestamp"
    df = df.reset_index()
    batch_data = []
    for _, row in df.iterrows():
        batch_data.append((row['timestamp'].strftime('%Y-%m-%d %H:%M:%S'),'DXY', float(row['Close'])))
    sql = "INSERT IGNORE INTO macro_indicators (timestamp, indicator_code, value) VALUES (%s, %s, %s)"
    curse.executemany(sql, batch_data)
    conn.commit()
except Exception as e:
    print(e)
finally:
    curse.close()
    conn.close()

# --- stage 1.3 (3/4): extraction of reddit sentiments ---
### Using JSON

In [1]:
import requests
from datetime import datetime,timezone
import re # regex

ref="https://www.google.com/"
headers = {
        # Identity: Must reflect Windows and a recent Chrome version
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        # Language: Tells the server the user's preferred language settings
        "Accept-Language": "en-US,en;q=0.9",

        # Connection Status: Standard practice for modern persistent connections
        "Connection": "keep-alive",

        # Previous website from which we referred this website
        "Referer": ref,

        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Cache-Control": "max-age=0",
        "Accept-Encoding": "gzip, deflate, br",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,webp,image/apng,*/*;q=0.8",

}
pattern = r"\b(bitcoin|ethereum|ether|dogecoin|btc|eth|doge|solana|sol|xrp|ripple|bnb|ada|cardano|dot|polkadot|litecoin|ltc|chainlink|link|ftx|luna|terra|avax|avalanche|matic|polygon)\b"

def insert_posts(symbol,timestamp,signal_text,senti_score,weight):
    inslist = [symbol,timestamp,signal_text,senti_score,weight]
    curse.execute("INSERT IGNORE INTO posts_logs (asset,timestamp,post_text,senti_score,weight) values (%s,%s,%s,%s,%s)",inslist)

# --- Getting .json response from reddit.json ---
response = requests.get("https://www.reddit.com/r/CryptoMarkets.json",headers=headers)

# --- Title Extraction from reddit posts ---
posts = response.json()['data']['children']
# data will take the values of the key 'data' from JSON dict
# children will return the list from the values : {'data':[d1,d2,d3]}
for p in posts:
    print(p['data']['title'])

    title = p['data'].get('title','No Title')# get title else write No Title
    body = p['data'].get('selftext',"")
    upvotes = max(0,p['data'].get("ups",0))
    signal_text = title if len(title)>30 else f"{title}. {body}"
    symbol = get_symbol(signal_text)
    unix_time = p['data'].get('created_utc')
    if symbol and is_question(title):
        senti_score = 0.5
        model_weight = upvotes*0.3
        dateob = datetime.fromtimestamp(unix_time,tz=timezone.utc)
        time = dateob.strftime('%Y-%m-%d %H:%M:%S')
        insert_posts(symbol,time,signal_text,senti_score,model_weight)

    elif symbol and not is_question(title):
        lemmatized_text = lemmatize_text(signal_text)
        senti_score = sentiment_analysis(lemmatized_text)
        model_weight = upvotes*1
        dateob = datetime.fromtimestamp(unix_time,tz=timezone.utc)
        time = dateob.strftime('%Y-%m-%d %H:%M:%S')
        senti_score = (senti_score+1)/2
        insert_posts(symbol,time,signal_text,senti_score,model_weight)
        conn.commit()

curse.close()
conn.close()





Daily Crypto Discussion - March 12, 2026


NameError: name 'get_symbol' is not defined

# --- stage 1.4 (4/4): extraction of crypto news ---
### Using Selenium

In [ ]:
from collections import defaultdict
from datetime import datetime,timedelta
from bs4 import BeautifulSoup
import re # regex
# --- coindesk extractor ---
# --- imports ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# --- chrome-options ---
chrome_options = Options()
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)
chrome_options.add_argument("--window-size=1200,800") # Set fixed, common viewport size
chrome_options.add_argument('--log-level=3') # Suppress unnecessary logging

# --- Navigating to the news website ---
service = Service(executable_path="C:/Users/dhair/OneDrive/Desktop/scrapper/chromedriver-win64/chromedriver-win64/chromedriver.exe")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),options=chrome_options) # this installs the chrome driver if versions mismatch
driver.get("https://www.coindesk.com/markets")
driver.execute_cdp_cmd('Network.setExtraHTTPHeaders',{'headers': headers})
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

# --- Extraction of headings and dates from source code ---
with open("extracted.html", "w", encoding="utf-8") as file:
    file.write(driver.page_source)
driver.close()
with open("extracted.html",encoding='utf-8') as extracted:
    soup = BeautifulSoup(extracted,'html.parser')
    heading_texts = [heading.text for heading in soup.find_all('h2')]

    # --- Headings Extraction ---
    headings = soup.find_all('h2')
    listh = []
    for heading in headings:
        listh.append(heading.text)

    # --- Dates Extraction ---
    paras = soup.find_all('p')
    texts=[]
    for para in paras:
        texts.extend([i.get_text() for i in para.find_all('span')])

    # --- Dates cleanup and formating ---
    dates = []
    for text in texts:
        dateobj = None
        if "minutes ago"  in text:
            dateobj =  datetime.now() - timedelta(minutes=int(text.split()[0]))
        elif "hours ago" in text:
            dateobj = datetime.now() - timedelta(hours=int(text.split()[0]))
        elif str(datetime.now().year) in text:
            text = text.replace(",","")
            dateobj = datetime.strptime(text,"%b %d %Y")
        else:
            continue
        if dateobj:
            db_format = dateobj.strftime('%Y-%m-%d %H:%M:%S')
            dates.append(db_format)

    # --- data clean-up ---
    pattern = r"bitcoin|ethereum|market|dogecoin|solana|sol|xrp"
    pattern2 = r"\b[A-Z]{3,5}\b"
    t = defaultdict(list)
    blacklist = ['ceo', 'cftc', 'market', 'etf', 'sec', 'usa','cnbc']
    for headings,time in zip(listh,dates):

        # --- Sentiment analysis ---
        found = re.findall(pattern,headings,re.IGNORECASE) + re.findall(pattern2,headings)
        symbols = set([s.lower() for s in found if s.lower() not in blacklist])
        for coin in symbols:
            lemm_text = lemmatize_text(headings)
            sentiment_score =  sentiment_analysis(lemm_text) # gives score between -4 to 4
            sentiment_score = (sentiment_score + 1)/2 # To keep the score between 0 and 1 because models such as scikit-learn and tensorflow etc. have the weight limits from 0 to 1
            t[coin].append({'headline':headings,'time':time,'sentiment score':sentiment_score})

    print(t)